# Week 09 — AI → LinuxCLI via MCP

This notebook demonstrates a **minimal MCP-like** (Model Context Protocol) service
implemented in **C** and called from a **Python** agent.

It builds directly on the Week 08 MCP server (port 9000) and shows how the same
pattern generalises to exposing **any** socket-based service as AI tools.

**Important:** Run in an isolated VM/container for lab work.  
The notebook writes two artefacts to `/mnt/data/`:
- `server.c` — educational MCP-like C server
- `client.py` — Python MCP client / agent example


## Learning Goals

1. Implement a socket-based server in C that understands newline-delimited JSON requests.
2. Advertise tools (with simple schemas) and execute safe, whitelisted Linux CLI actions.
3. Build a Python client/agent that performs `initialize`, `list_tools`, and `call_tool`.
4. Understand security trade-offs and how to harden the service for labs.


## What is MCP? (Concise)

- MCP (Model Context Protocol) is an application-level JSON-based protocol for AI agents
  to discover and call external tools.
- Messages are typically newline-delimited JSON objects over streams (TCP / WebSocket / stdio).
- Key message types: `initialize`, `list_tools`, `call_tool`, `notifications` / `progress`.

This notebook implements a minimal MCP-like subset (`initialize` + `list_tools` + `call_tool`)
focused on mapping AI tool calls to Linux CLI commands while teaching socket programming.


## Handshake & Tool Call Flow

```
Client                     MCP Server
------                     ----------
   ---> initialize  -------------------->
   <--- initialize_result ---------------
   ---> list_tools --------------------->
   <--- list_tools_result --------------
   ---> call_tool(list_files) ----------->
   <--- tool_result ----------------------
```

Messages are newline-delimited JSON objects.  
Each request carries an `id` that the server echoes in its response for correlation.


## Implementation Overview

- **Server language:** C (sockets, fork, basic argument handling).
- **Client language:** Python (agent example + mocked LLM).
- **JSON parsing:** The starter server uses simple string extraction for clarity;
  replace with `cJSON` or `jsmn` in production labs.
- **Security:** Server whitelists tools and sanitises arguments.  
  Still run in an isolated environment — production requires TLS + auth + robust parsing.


In [1]:
# Write the educational C server to /mnt/data/server.c
server_c = r'''
/*
 * server.c
 * Minimal MCP-like TCP server for teaching.
 * - Accepts a client and forks a child to serve it.
 * - Newline-delimited JSON messages.
 * - Implements: initialize, list_tools, call_tool
 *   (list_files, get_time, delete_older_than_days)
 *
 * WARNING: Educational code.
 * Replace naive JSON handling with cJSON/jsmn for production.
 */

#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <errno.h>
#include <arpa/inet.h>
#include <time.h>
#include <sys/types.h>
#include <sys/socket.h>
#include <sys/wait.h>
#include <ctype.h>

#define PORT    9000
#define BACKLOG 4
#define BUFSIZE 16384
#define MAX_ARG 512

int send_json(int fd, const char *json) {
    size_t len = strlen(json);
    if (write(fd, json, len) != (ssize_t)len) return -1;
    if (write(fd, "\n", 1) != 1) return -1;
    return 0;
}

void sanitize_path(const char *in, char *out, size_t outsz) {
    size_t i = 0;
    for (; *in && i + 1 < outsz; ++in) {
        if (isalnum((unsigned char)*in) ||
            *in == '/' || *in == '.' ||
            *in == '_' || *in == '-' || *in == ' ')
            out[i++] = *in;
    }
    out[i] = 0;
}

int extract_field(const char *buf, const char *field,
                  char *out, size_t outsz) {
    const char *p = strstr(buf, field);
    if (!p) return 0;
    p = strchr(p, ':');
    if (!p) return 0;
    p++;
    while (*p && (*p == ' ' || *p == '"')) ++p;
    size_t i = 0;
    while (*p && *p != '"' && *p != ',' && *p != '}' &&
           *p != '\n' && i + 1 < outsz)
        out[i++] = *p++;
    out[i] = 0;
    while (i > 0 && out[i-1] == ' ') out[--i] = 0;
    return 1;
}

void tool_list_files(int cfd, const char *params) {
    char rawpath[MAX_ARG] = ".";
    extract_field(params, "\"path\"", rawpath, sizeof(rawpath));
    char path[MAX_ARG];
    sanitize_path(rawpath, path, sizeof(path));
    if (!strlen(path)) strcpy(path, ".");
    char cmd[2048];
    snprintf(cmd, sizeof(cmd), "ls -la --color=never %s 2>&1", path);
    FILE *fp = popen(cmd, "r");
    if (!fp) {
        send_json(cfd,
            "{\"id\":null,\"type\":\"error\",\"error\":\"ls failed\"}");
        return;
    }
    send_json(cfd,
        "{\"id\":null,\"type\":\"notification\","
        "\"event\":\"tool_progress\",\"message\":\"listing files\"}");
    char output[BUFSIZE]; output[0] = 0;
    char line[1024];
    while (fgets(line, sizeof(line), fp)) {
        for (char *q = line; *q; ++q)
            if (*q == '"' || *q == '\\') *q = ' ';
        strncat(output, line, sizeof(output)-strlen(output)-1);
    }
    pclose(fp);
    char out[BUFSIZE * 2];
    snprintf(out, sizeof(out),
        "{\"id\":1,\"type\":\"response\","
        "\"result\":{\"tool\":\"list_files\",\"output\":\"%s\"}}",
        output);
    send_json(cfd, out);
}

void tool_get_time(int cfd) {
    time_t t = time(NULL);
    struct tm tm = *localtime(&t);
    char buf[200];
    snprintf(buf, sizeof(buf),
        "%04d-%02d-%02d %02d:%02d:%02d",
        tm.tm_year+1900, tm.tm_mon+1, tm.tm_mday,
        tm.tm_hour, tm.tm_min, tm.tm_sec);
    char out[512];
    snprintf(out, sizeof(out),
        "{\"id\":1,\"type\":\"response\","
        "\"result\":{\"tool\":\"get_time\",\"time\":\"%s\"}}",
        buf);
    send_json(cfd, out);
}

void tool_delete_older(int cfd, const char *params) {
    char rawpath[MAX_ARG] = ".";
    char days_s[32] = "0";
    extract_field(params, "\"path\"", rawpath, sizeof(rawpath));
    extract_field(params, "\"days\"", days_s, sizeof(days_s));
    char path[MAX_ARG];
    sanitize_path(rawpath, path, sizeof(path));
    int days = atoi(days_s);
    if (days <= 0) {
        send_json(cfd,
            "{\"id\":1,\"type\":\"response\","
            "\"result\":{\"error\":\"invalid days value\"}}");
        return;
    }
    char cmd[2048];
    snprintf(cmd, sizeof(cmd),
        "find %s -maxdepth 1 -type f -mtime +%d -print -delete 2>&1",
        path, days);
    FILE *fp = popen(cmd, "r");
    if (!fp) {
        send_json(cfd,
            "{\"id\":null,\"type\":\"error\",\"error\":\"find failed\"}");
        return;
    }
    char output[BUFSIZE]; output[0] = 0;
    char line[1024];
    while (fgets(line, sizeof(line), fp)) {
        for (char *q = line; *q; ++q)
            if (*q == '"' || *q == '\\') *q = ' ';
        strncat(output, line, sizeof(output)-strlen(output)-1);
    }
    pclose(fp);
    char out[BUFSIZE * 2];
    snprintf(out, sizeof(out),
        "{\"id\":1,\"type\":\"response\","
        "\"result\":{\"tool\":\"delete_older_than_days\","
        "\"output\":\"%s\"}}",
        output);
    send_json(cfd, out);
}

int main(void) {
    int sockfd, newfd;
    struct sockaddr_in serv, cli;
    socklen_t sin_size;
    char buf[BUFSIZE];
    int yes = 1;

    sockfd = socket(AF_INET, SOCK_STREAM, 0);
    setsockopt(sockfd, SOL_SOCKET, SO_REUSEADDR, &yes, sizeof(int));
    serv.sin_family = AF_INET;
    serv.sin_addr.s_addr = INADDR_ANY;
    serv.sin_port = htons(PORT);
    memset(&(serv.sin_zero), 0, 8);
    bind(sockfd, (struct sockaddr *)&serv, sizeof(struct sockaddr));
    listen(sockfd, BACKLOG);
    printf("MCP-like server on port %d\n", PORT);

    while (1) {
        sin_size = sizeof(struct sockaddr_in);
        newfd = accept(sockfd, (struct sockaddr *)&cli, &sin_size);
        printf("Client: %s\n", inet_ntoa(cli.sin_addr));
        pid_t pid = fork();
        if (pid == 0) {
            close(sockfd);
            ssize_t nb;
            while ((nb = read(newfd, buf, BUFSIZE-1)) > 0) {
                buf[nb] = 0;
                if (strstr(buf, "\"method\":\"initialize\"") ||
                    strstr(buf, "\"method\": \"initialize\"")) {
                    send_json(newfd,
                        "{\"id\":1,\"type\":\"response\","
                        "\"result\":{\"server\":\"LinuxCLI MCP Server\","
                        "\"version\":\"1.0\"}}");
                } else if (strstr(buf, "\"method\":\"list_tools\"") ||
                           strstr(buf, "\"method\": \"list_tools\"")) {
                    send_json(newfd,
                        "{\"id\":1,\"type\":\"response\","
                        "\"result\":{\"tools\":["
                        "{\"name\":\"list_files\","
                         "\"desc\":\"List files in a directory\","
                         "\"schema\":{\"path\":\"string\"}},"
                        "{\"name\":\"get_time\","
                         "\"desc\":\"Get server time\","
                         "\"schema\":{}},"
                        "{\"name\":\"delete_older_than_days\","
                         "\"desc\":\"Delete files older than N days\","
                         "\"schema\":{\"path\":\"string\","
                                      "\"days\":\"integer\"}}]}}");
                } else if (strstr(buf, "\"method\":\"call_tool\"") ||
                           strstr(buf, "\"method\": \"call_tool\"")) {
                    if      (strstr(buf, "\"list_files\""))
                        tool_list_files(newfd, buf);
                    else if (strstr(buf, "\"get_time\""))
                        tool_get_time(newfd);
                    else if (strstr(buf, "\"delete_older_than_days\""))
                        tool_delete_older(newfd, buf);
                    else
                        send_json(newfd,
                            "{\"id\":1,\"type\":\"response\","
                            "\"result\":{\"error\":\"unknown tool\"}}");
                } else {
                    send_json(newfd,
                        "{\"id\":null,\"type\":\"error\","
                        "\"error\":\"unknown method\"}");
                }
            }
            close(newfd);
            exit(0);
        } else if (pid > 0) {
            close(newfd);
            while (waitpid(-1, NULL, WNOHANG) > 0) {}
        } else {
            perror("fork"); close(newfd);
        }
    }
    return 0;
}
'''

import os
os.makedirs('/mnt/data', exist_ok=True)
open('/mnt/data/server.c', 'w').write(server_c)
print('Wrote /mnt/data/server.c')

PermissionError: [Errno 13] Permission denied: '/mnt/data'

### Compile the server

```bash
cd /mnt/data
gcc server.c -o mcp_server -std=c11
./mcp_server &
```

If `gcc` is not available, run in a container/VM with build tools installed.


In [ ]:
# Write the Python client to /mnt/data/client.py
client_py = '''
# client.py — minimal MCP client
import socket, json

HOST, PORT = '127.0.0.1', 9000

def send_msg(s, obj):
    s.send((json.dumps(obj) + '\\n').encode())

def recv_msg(s, timeout=5.0):
    s.settimeout(timeout)
    data = b''
    try:
        while True:
            chunk = s.recv(4096)
            if not chunk: break
            data += chunk
            if b'\\n' in chunk: break
    except socket.timeout:
        pass
    if not data: return None
    try:
        return json.loads(data.decode().strip())
    except Exception:
        return data.decode()

if __name__ == '__main__':
    s = socket.socket()
    s.connect((HOST, PORT))
    send_msg(s, {'id':1, 'method':'initialize', 'params':{}})
    print('init   ->', recv_msg(s))
    send_msg(s, {'id':2, 'method':'list_tools', 'params':{}})
    print('tools  ->', recv_msg(s))
    send_msg(s, {'id':3, 'method':'call_tool',
                 'params':{'tool':'list_files','args':{'path':'.'}}})
    print('files  ->', recv_msg(s))
    send_msg(s, {'id':4, 'method':'call_tool',
                 'params':{'tool':'get_time','args':{}}})
    print('time   ->', recv_msg(s))
    s.close()
'''

open('/mnt/data/client.py', 'w').write(client_py)
print('Wrote /mnt/data/client.py')

### Run the client

```bash
# (server must be running first)
python3 /mnt/data/client.py
```

Expected output: JSON responses for `initialize`, `list_tools`, `list_files`, `get_time`.


## Mocked AI Agent Example

The cell below shows how an agent discovers tools with `list_tools`,
decides which tool to call (here mocked — no API key required),
then forwards `call_tool` to the MCP server.


In [ ]:
import socket, json

HOST, PORT = '127.0.0.1', 9000

def send_msg(s, obj):
    s.send((json.dumps(obj) + '\n').encode())

def recv_msg(s, timeout=5.0):
    s.settimeout(timeout)
    data = b''
    try:
        while True:
            chunk = s.recv(4096)
            if not chunk: break
            data += chunk
            if b'\n' in chunk: break
    except socket.timeout:
        pass
    if not data: return None
    try:
        return json.loads(data.decode().strip())
    except Exception:
        return data.decode()

def mocked_model_decision(user_request, tools):
    """Tiny heuristic mock — replace with real LLM call in production."""
    lower = user_request.lower()
    if 'time' in lower:
        return {'tool': 'get_time', 'args': {}}
    if 'delet' in lower or 'old' in lower:
        return {'tool': 'delete_older_than_days', 'args': {'path': '.', 'days': 30}}
    return {'tool': 'list_files', 'args': {'path': '.'}}

def agent_flow(user_request):
    s = socket.socket()
    s.connect((HOST, PORT))

    # 1. Discover tools
    send_msg(s, {'id': 1, 'method': 'list_tools', 'params': {}})
    tools = recv_msg(s)
    print('Available tools:', [t['name'] for t in tools['result']['tools']])

    # 2. Model decides (mocked)
    decision = mocked_model_decision(user_request, tools)
    print(f'Model chose: {decision["tool"]}({decision["args"]})')

    # 3. Call tool
    send_msg(s, {'id': 10, 'method': 'call_tool',
                 'params': {'tool': decision['tool'], 'args': decision['args']}})
    result = recv_msg(s)
    print('Result:', result)
    s.close()

# Demo (MCP server must be running on port 9000)
# agent_flow('List all source files in the current directory')
# agent_flow('What time is it on the server?')
print('Agent function defined. Start mcp_server then call agent_flow(...).')

## Calling with a Real LLM (OpenAI)

Run the MCP server locally, then call it via the OpenAI API:

```python
import openai, socket, json

client = openai.OpenAI()  # uses OPENAI_API_KEY env var

# Build OpenAI tool schemas from list_tools response
tools_resp = mcp_list_tools()   # call to MCP server
openai_tools = [
    {
        'type': 'function',
        'function': {
            'name': t['name'],
            'description': t['desc'],
            'parameters': {
                'type': 'object',
                'properties': {k: {'type': v}
                               for k, v in t.get('schema', {}).items()},
            },
        },
    }
    for t in tools_resp['result']['tools']
]

# Agent loop: NL → OpenAI → tool call → MCP → answer
messages = [{'role': 'user', 'content': 'List files and tell me the time.'}]
response = client.chat.completions.create(
    model='gpt-4o', tools=openai_tools, messages=messages
)
# dispatch tool calls back to MCP server ...
```

Or using the OpenAI CLI:

```bash
openai api chat.completions.create \
    --model gpt-4o \
    --mcp-server http://localhost:9000 \
    -m "List files and tell me the server time."
```


## Lab Assignments & Grading Rubric

**Lab 1 — Minimal MCP Server (40%)**  
- Starting point: `/mnt/data/server.c` (written by this notebook).  
- Requirements: `initialize`, `list_tools`, `call_tool` for `list_files` and `get_time`.  
- Demonstrate with `/mnt/data/client.py`.

**Lab 2 — Additional Tool (30%)**  
- Implement `delete_older_than_days` safely.  
- Validate arguments; add a `dry_run` parameter that lists without deleting.

**Lab 3 — Agent Integration (30%)**  
- Build an agent: `list_tools` → choose tool → `call_tool` → format result.  
- Mocked LLM is acceptable; real LLM with `OPENAI_API_KEY` for extra credit.

Extra credit: TLS + token auth, replace naive parsing with `cJSON`/`jsmn`,
sandbox execution (namespaces, chroot, containers).


## Security & Operational Notes

| Concern | Status in this notebook | Production fix |
|---------|------------------------|----------------|
| Auth | None — any client connects | Token header or mTLS |
| JSON parsing | Naive `strstr` — fragile | `cJSON` / `jsmn` |
| Path sanitisation | `sanitize_path` (basic) | `realpath` + allowlist |
| Transport | Plain TCP | TLS (`openssl` or `mbedtls`) |
| Process isolation | `fork()` per request | Container / namespace |

**Run the server in an isolated VM or container for all lab exercises.**


## In-Class Demo Plan (10–15 min)

1. Compile and start the server: `gcc server.c -o mcp_server && ./mcp_server &`
2. Run `client.py` to show `initialize` → `list_tools` → `list_files` → `get_time`.
3. Run the mocked agent cell above — show NL → tool selection → execution.
4. Discussion: how this relates to Week 08's FTP assignment, and why MCP is the
   next layer up (structured schema, AI-callable, protocol-neutral).


## Next Steps

- Replace string parsing with `cJSON` / `jsmn` and add schema validation.
- Add WebSocket transport for browser-based agents.
- Wire in a real LLM and show the full Sense → Decide → Act loop.
- Extend tools to expose GPIO / sensors for robotics and IoT labs.
